In [2]:
### Feature Engineering ###
# Top N selling products per category  (by units_sold)    -> product_sales

from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv('data/sales.csv')
df_features = df.copy()

le_cat = LabelEncoder()
le_prod = LabelEncoder()

# Feature Table: Product-level (top N products per category)
product_sales = (
  df_features.groupby(['date', 'product_category', 'product_name'])[['units_sold', 'total_revenue']]
  .sum()
  .reset_index()
  .sort_values(['product_category', 'product_name', 'date'])
)

product_sales['category_encoded'] = le_cat.fit_transform(product_sales['product_category'])
product_sales['product_encoded']  = le_prod.fit_transform(product_sales['product_name'])
product_sales['date'] = pd.to_datetime(product_sales['date'], errors='coerce')

# Temporal features
product_sales['year']        = product_sales['date'].dt.year
product_sales['month']       = product_sales['date'].dt.month
product_sales['week']        = product_sales['date'].dt.isocalendar().week
product_sales['day_of_week'] = product_sales['date'].dt.dayofweek

# Lag features on units_sold per product (daily)
for lag in [1, 7, 14, 30]:
  product_sales[f'units_lag_{lag}'] = product_sales.groupby('product_name')['units_sold'].shift(lag)

product_sales = product_sales.dropna(subset=[f'units_lag_{lag}' for lag in [1, 7, 14, 30]])
print(f"product_sales rows: {len(product_sales)}")
# display(product_sales.head())

product_sales rows: 33600


In [3]:
### Train/Test Data Setup ###
# 80% train / 20% test split for each prediction target.

# Split: Top N products per category (by units_sold)
feature_columns = [
    'category_encoded', 'product_encoded', 'year', 'month', 'week', 'day_of_week',
    'units_lag_1', 'units_lag_7', 'units_lag_14', 'units_lag_30'
]
target = 'units_sold'

product_sales_sorted = product_sales.sort_values('date')
train_size_p = int(len(product_sales_sorted) * 0.8)
product_train_df = product_sales_sorted.iloc[:train_size_p]
product_test_df  = product_sales_sorted.iloc[train_size_p:]

X_train_product = product_train_df[feature_columns]
y_train_product = product_train_df[target]
X_test_product  = product_test_df[feature_columns]
y_test_product  = product_test_df[target]

print(f"Product model — train: {len(X_train_product)}, test: {len(X_test_product)}")

Product model — train: 26880, test: 6720


In [ ]:
### Model Selection, Training, and Evaluation ###
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import lightgbm as lgb
import numpy as np

### LGB ### A predition model for product sales per category (predicts units_sold)
lgb_params = dict(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=432)
model_lgb_product = lgb.LGBMRegressor(**lgb_params)
model_lgb_product.fit(X_train_product, y_train_product)     # training

predictions_lgb = model_lgb_product.predict(X_test_product) # prediction
mae_lgb = mean_absolute_error(y_test_product, predictions_lgb)
mse_lgb = mean_squared_error(y_test_product, predictions_lgb)
rmse_lgb = np.sqrt(mse_lgb)
r2_lgb = r2_score(y_test_product, predictions_lgb)

print(f"MAE  (LightGBM): {mae_lgb:.4f}")
print(f"RMSE (LightGBM): {rmse_lgb:.4f}")
print(f"R2   (LightGBM): {r2_lgb:.4f}")

### XGB ### A predition model for product sales per category (predicts units_sold)
xgb_params = dict(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=432)
model_xgb_product = XGBRegressor(**xgb_params)
model_xgb_product.fit(X_train_product, y_train_product)

predictions_xgb = model_xgb_product.predict(X_test_product)

mae_xgb = mean_absolute_error(y_test_product, predictions_xgb)
mse_xgb = mean_squared_error(y_test_product, predictions_xgb)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_test_product, predictions_xgb)

print(f"MAE  (XGB): {mae_xgb:.4f}")
print(f"RMSE (XGB): {rmse_xgb:.4f}")
print(f"R2   (XGB): {r2_xgb:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000281 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 250
[LightGBM] [Info] Number of data points in the train set: 26880, number of used features: 10
[LightGBM] [Info] Start training from score 7.729725
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [11]:
### Save Models to Artifactory ### 

import joblib

# Bundle each model with the LabelEncoder so consumers always use
# the exact same category→integer mapping that was used during training.
lgb_artifact = {
  'model': model_lgb_product,
  'le_cat': le_cat,
  'le_prod': le_prod,
  'feature_columns': feature_columns,
}

joblib.dump(lgb_artifact, 'models/top_products_prediction_lgb.joblib')

['models/top_products_prediction_lgb.joblib']